# Lab Assignment 6 – CNN for Automated Diagnosis from Chest X-Rays
**Course:** CSET343 – AI in Healthcare | **B.Tech, 4th Year, Sem VII**

**Objective:** Design and evaluate CNN models (plus classical classifiers on hand-crafted features) for automated diagnosis from chest X-ray images.

> **Note:** The assignment text says "heart disease", but the specified dataset (Chest X-Ray Images, ~5,863 images, Normal vs Pneumonia) is the Kaggle *Chest X-Ray Pneumonia* dataset. This notebook therefore solves the binary **Normal vs Pneumonia** task (Pneumonia = positive class = 1).

**Pipeline**
1. Data acquisition & exploration
2. Preprocessing, feature extraction (HOG, LBP, GLCM), 70/15/15 split
3. Models: CNN (with CV hyper-parameter tuning) + SVM / Logistic Regression / Random Forest on features
4. Evaluation: Accuracy, Precision, Recall, F1, AUROC, confusion matrices, error analysis, comparison

## 0. Setup

In [ ]:
# Run once if needed:
# !pip install -q numpy pandas matplotlib seaborn scikit-learn scikit-image pillow joblib tensorflow kagglehub

import os, random, warnings, itertools, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from joblib import Parallel, delayed

from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve, classification_report)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = 128
CLASSES = ["NORMAL", "PNEUMONIA"]          # label 0 = Normal, 1 = Pneumonia (positive class)
sns.set_style("whitegrid")
print("TensorFlow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## Task 1: Data Acquisition and Exploration
### 1.1 Download and load the dataset
The dataset is *Chest X-Ray Images (Pneumonia)* by Paul Mooney on Kaggle. Option A downloads it with `kagglehub` (needs internet; Kaggle credentials may be required). Option B: download manually, unzip, and set `DATA_DIR` to the folder that contains `train/`, `test/`, `val/`.

In [ ]:
DATA_DIR = None   # <-- Option B: set manually, e.g. r"C:\data\chest_xray"  or "/content/chest_xray"

if DATA_DIR is None:                       # Option A: automatic download
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Dataset root:", DATA_DIR)

In [ ]:
# The original dataset ships with train/val/test folders (val has only 16 images).
# We POOL every image and create our own stratified 70/15/15 split later (Task 2).
records = []
for root, dirs, files in os.walk(DATA_DIR):
    cls = os.path.basename(root).upper()
    if cls in CLASSES:
        orig_split = os.path.basename(os.path.dirname(root)).lower()
        for f in files:
            if f.lower().endswith((".jpeg", ".jpg", ".png")):
                records.append((os.path.join(root, f), CLASSES.index(cls), cls, orig_split))

df = pd.DataFrame(records, columns=["path", "label", "class", "orig_split"])
# if the archive contains a duplicated nested copy (e.g. chest_xray/chest_xray), drop duplicate files
df["fname"] = df["path"].apply(os.path.basename)
df = df.drop_duplicates(subset=["fname", "class"]).drop(columns="fname").reset_index(drop=True)
print("Total images found:", len(df))
df.head()

### 1.2 Number of images per class (class balance)

In [ ]:
counts = df["class"].value_counts().reindex(CLASSES)
print(counts.to_string())
print(f"\nPneumonia : Normal ratio = {counts['PNEUMONIA']/counts['NORMAL']:.2f} : 1")
print(f"Majority-class baseline accuracy = {counts.max()/counts.sum():.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=counts.index, y=counts.values, palette=["#4C9BE8", "#E8624C"], ax=ax[0])
for i, v in enumerate(counts.values): ax[0].text(i, v + 20, str(v), ha="center", fontweight="bold")
ax[0].set_title("Images per class (all data)"); ax[0].set_ylabel("count")
ax[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%", colors=["#4C9BE8", "#E8624C"], startangle=90)
ax[1].set_title("Class proportion")
plt.tight_layout(); plt.show()

print("\nOriginal Kaggle split distribution:")
print(pd.crosstab(df["orig_split"], df["class"]))

**Observation:** The dataset is **imbalanced** – Pneumonia images are roughly 2.9× more numerous than Normal ones. Accuracy alone is therefore misleading (a model predicting "Pneumonia" always already scores ≈ 73 %), so we also report Precision, Recall, F1 and AUROC, use *stratified* splits, and apply *class weights* during training.

### 1.3 Display random sample images from each class

In [ ]:
def show_samples(n=8, seed=SEED):
    rng = np.random.RandomState(seed)
    fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 5.2))
    for r, cls in enumerate(CLASSES):
        sub = df[df["class"] == cls]
        pick = sub.iloc[rng.choice(len(sub), n, replace=False)]
        for c, p in enumerate(pick["path"]):
            img = Image.open(p).convert("L")
            axes[r, c].imshow(img, cmap="gray"); axes[r, c].axis("off")
            axes[r, c].set_title(f"{cls}\n{img.size[0]}x{img.size[1]}", fontsize=8)
    plt.suptitle("Random samples: top = NORMAL, bottom = PNEUMONIA", fontsize=13)
    plt.tight_layout(); plt.show()

show_samples(8)

In [ ]:
# Raw image-size variability (justifies resizing)
sizes = [Image.open(p).size for p in df["path"].sample(300, random_state=SEED)]
w, h = zip(*sizes)
print(f"Width : min={min(w)}, max={max(w)}, mean={np.mean(w):.0f}")
print(f"Height: min={min(h)}, max={max(h)}, mean={np.mean(h):.0f}")

## Task 2: Data Preprocessing
### 2.1 Resize to 128×128 and convert to grayscale
Every image is converted to single-channel grayscale (some files are stored as RGB although they are visually grayscale), resized to 128×128 and stored as `uint8`.

In [ ]:
def load_img(path):
    img = Image.open(path).convert("L")                    # grayscale
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)  # fixed size
    return np.asarray(img, dtype=np.uint8)

t0 = time.time()
X_img = np.stack(Parallel(n_jobs=-1)(delayed(load_img)(p) for p in df["path"]))
y = df["label"].values
print("Image tensor:", X_img.shape, X_img.dtype, f"| {time.time()-t0:.0f}s")
print("Pixel range:", X_img.min(), "-", X_img.max())

### 2.2 Feature extraction: HOG, LBP, GLCM
| Method | What it captures | Settings |
|---|---|---|
| **HOG** | Edge/gradient orientation → shapes, lung borders, opacity edges | 9 orientations, 16×16 cells, 2×2 blocks |
| **LBP** | Local micro-texture (uniform patterns) | P=8, R=1, `uniform` → 10-bin histogram |
| **GLCM** | Second-order statistics of gray-level pairs | 32 gray levels, distances {1,3}, angles {0°,45°,90°,135°}; contrast, dissimilarity, homogeneity, energy, correlation, ASM |

In [ ]:
LBP_P, LBP_R = 8, 1
GLCM_LEVELS = 32
GLCM_DIST   = [1, 3]
GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]
GLCM_PROPS  = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]

def hog_feat(img):
    return hog(img, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2),
               block_norm="L2-Hys", feature_vector=True)

def lbp_feat(img):
    lbp = local_binary_pattern(img, LBP_P, LBP_R, method="uniform")
    hist, _ = np.histogram(lbp, bins=LBP_P + 2, range=(0, LBP_P + 2), density=True)
    return hist

def glcm_feat(img):
    q = (img // (256 // GLCM_LEVELS)).astype(np.uint8)       # quantise 256 -> 32 levels
    g = graycomatrix(q, GLCM_DIST, GLCM_ANGLES, levels=GLCM_LEVELS, symmetric=True, normed=True)
    return np.concatenate([graycoprops(g, p).ravel() for p in GLCM_PROPS])

def extract_all(img):
    return hog_feat(img), lbp_feat(img), glcm_feat(img)

t0 = time.time()
out = Parallel(n_jobs=-1)(delayed(extract_all)(im) for im in X_img)
F_hog  = np.array([o[0] for o in out])
F_lbp  = np.array([o[1] for o in out])
F_glcm = np.array([o[2] for o in out])
# 2.3 Combined feature vector per image
F_all = np.hstack([F_hog, F_lbp, F_glcm]).astype(np.float32)

print(f"HOG : {F_hog.shape[1]} features")
print(f"LBP : {F_lbp.shape[1]} features")
print(f"GLCM: {F_glcm.shape[1]} features")
print(f"Combined feature matrix: {F_all.shape}  ({time.time()-t0:.0f}s)")

In [ ]:
# Visualise what each descriptor "sees" on one Normal and one Pneumonia image
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for r, cls in enumerate([0, 1]):
    i = np.where(y == cls)[0][0]; im = X_img[i]
    _, hog_im = hog(im, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), visualize=True)
    lbp_im = local_binary_pattern(im, LBP_P, LBP_R, method="uniform")
    axes[r, 0].imshow(im, cmap="gray");       axes[r, 0].set_title(f"{CLASSES[cls]} (128x128)")
    axes[r, 1].imshow(hog_im, cmap="gray");   axes[r, 1].set_title("HOG")
    axes[r, 2].imshow(lbp_im, cmap="nipy_spectral"); axes[r, 2].set_title("LBP (uniform)")
    for a in axes[r]: a.axis("off")
plt.tight_layout(); plt.show()

# GLCM feature differences between classes
glcm_names = [f"{p}_d{d}_a{int(np.degrees(a))}" for p in GLCM_PROPS for d in GLCM_DIST for a in GLCM_ANGLES]
gdf = pd.DataFrame(F_glcm, columns=glcm_names); gdf["class"] = df["class"].values
sel = [f"{p}_d1_a0" for p in GLCM_PROPS]
gdf.groupby("class")[sel].mean().T.round(4)

### 2.4 Train / Validation / Test split (70 / 15 / 15, stratified)

In [ ]:
idx = np.arange(len(y))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
idx_val, idx_test  = train_test_split(idx_tmp, test_size=0.50, stratify=y[idx_tmp], random_state=SEED)

split_tbl = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Images": [len(idx_train), len(idx_val), len(idx_test)],
    "Normal": [(y[i] == 0).sum() for i in (idx_train, idx_val, idx_test)],
    "Pneumonia": [(y[i] == 1).sum() for i in (idx_train, idx_val, idx_test)],
})
split_tbl["Fraction"] = (split_tbl["Images"] / len(y)).round(3)
display(split_tbl)

# Feature-vector splits (classical ML)
scaler = StandardScaler().fit(F_all[idx_train])              # fit on TRAIN only -> no leakage
Xf_train, Xf_val, Xf_test = (scaler.transform(F_all[i]) for i in (idx_train, idx_val, idx_test))
# Image splits (CNN) scaled to [0,1], channel axis added
Xi_train, Xi_val, Xi_test = ((X_img[i].astype("float32") / 255.0)[..., None] for i in (idx_train, idx_val, idx_test))
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]
print("Feature shapes:", Xf_train.shape, Xf_val.shape, Xf_test.shape)
print("Image   shapes:", Xi_train.shape, Xi_val.shape, Xi_test.shape)

## Task 3: Model Training and Basic Evaluation
### 3.0 Evaluation helpers
Positive class = **Pneumonia**. In a medical-screening context **Recall (sensitivity)** matters a lot: a false negative is a missed sick patient.

In [ ]:
results = {}        # name -> dict of metrics (on TEST), plus probabilities
val_scores = {}     # name -> validation AUROC / F1 (used for model selection)

def evaluate(name, y_true, prob, thr=0.5, store=True, split="Test"):
    pred = (prob >= thr).astype(int)
    m = dict(Accuracy=accuracy_score(y_true, pred),
             Precision=precision_score(y_true, pred, zero_division=0),
             Recall=recall_score(y_true, pred),
             F1=f1_score(y_true, pred),
             AUROC=roc_auc_score(y_true, prob))
    if store:
        results[name] = dict(metrics=m, prob=prob, pred=pred)
    print(f"[{name} | {split}] " + "  ".join(f"{k}={v:.4f}" for k, v in m.items()))
    return m

def plot_cm(name, y_true, pred, ax=None, cmap="Blues"):
    cm = confusion_matrix(y_true, pred)
    ax = ax or plt.gca()
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, cbar=False, ax=ax,
                xticklabels=["Pred Normal", "Pred Pneumonia"], yticklabels=["True Normal", "True Pneumonia"])
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"{name}\nFP={fp}  FN={fn}")

### 3.1 Baseline CNN
**Architecture:** 3 × [Conv → BatchNorm → ReLU → MaxPool] (32, 64, 128 filters) → Global Average Pooling → Dense(64) → Dropout → Sigmoid.
Light on-the-fly augmentation (small rotation / zoom / shift) reduces over-fitting; **class weights** counter the 1 : 2.9 imbalance.

In [ ]:
def build_cnn(lr=1e-3, dropout=0.4, base_filters=32):
    f = base_filters
    m = keras.Sequential([
        layers.Input((IMG_SIZE, IMG_SIZE, 1)),
        layers.RandomRotation(0.05), layers.RandomZoom(0.10), layers.RandomTranslation(0.05, 0.05),
        layers.Conv2D(f, 3, padding="same"),     layers.BatchNormalization(), layers.ReLU(), layers.MaxPooling2D(),
        layers.Conv2D(f*2, 3, padding="same"),   layers.BatchNormalization(), layers.ReLU(), layers.MaxPooling2D(),
        layers.Conv2D(f*4, 3, padding="same"),   layers.BatchNormalization(), layers.ReLU(), layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
              metrics=["accuracy", keras.metrics.AUC(name="auc")])
    return m

cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
CLASS_WEIGHT = {0: cw[0], 1: cw[1]}
print("Class weights:", {k: round(v, 3) for k, v in CLASS_WEIGHT.items()})
build_cnn().summary()

### 3.2 Hyper-parameter tuning with Cross-Validation
Stratified **3-fold CV on the training set only** over a grid of learning rate × dropout × base filters (validation/test sets are untouched). Each configuration is trained for a small number of epochs to keep the search tractable; the criterion is the mean **validation AUROC** across folds.
Reduce the grid or `CV_EPOCHS` if you are on CPU only.

In [ ]:
param_grid = {"lr": [1e-3, 3e-4], "dropout": [0.3, 0.5], "base_filters": [16, 32]}
CV_FOLDS, CV_EPOCHS, BATCH = 3, 6, 32

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
cv_rows = []
keys = list(param_grid)
t0 = time.time()
for combo in itertools.product(*param_grid.values()):
    params = dict(zip(keys, combo)); fold_auc, fold_f1 = [], []
    for tr, va in skf.split(Xi_train, y_train):
        keras.backend.clear_session()
        m = build_cnn(**params)
        m.fit(Xi_train[tr], y_train[tr], epochs=CV_EPOCHS, batch_size=BATCH,
              class_weight=CLASS_WEIGHT, verbose=0)
        p = m.predict(Xi_train[va], verbose=0).ravel()
        fold_auc.append(roc_auc_score(y_train[va], p)); fold_f1.append(f1_score(y_train[va], p >= 0.5))
    cv_rows.append({**params, "cv_AUROC": np.mean(fold_auc), "cv_AUROC_std": np.std(fold_auc), "cv_F1": np.mean(fold_f1)})
    print(params, f"-> AUROC {np.mean(fold_auc):.4f} ± {np.std(fold_auc):.4f} | F1 {np.mean(fold_f1):.4f}")

cv_df = pd.DataFrame(cv_rows).sort_values("cv_AUROC", ascending=False).reset_index(drop=True)
best_params = {k: cv_df.loc[0, k] for k in keys}
best_params["base_filters"] = int(best_params["base_filters"])
print(f"\nCV finished in {(time.time()-t0)/60:.1f} min.  Best params: {best_params}")
display(cv_df.round(4))

### 3.3 Train the final CNN with the best hyper-parameters
Early stopping and LR reduction monitor the **validation set**; the best epoch's weights are restored.

In [ ]:
keras.backend.clear_session()
cnn = build_cnn(**best_params)
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]
hist = cnn.fit(Xi_train, y_train, validation_data=(Xi_val, y_val), epochs=30, batch_size=BATCH,
               class_weight=CLASS_WEIGHT, callbacks=callbacks, verbose=2)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, (k, t) in zip(ax, [("loss", "Loss"), ("accuracy", "Accuracy"), ("auc", "AUROC")]):
    a.plot(hist.history[k], label="train"); a.plot(hist.history["val_" + k], label="validation")
    a.set_title(f"CNN {t}"); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

p_val_cnn  = cnn.predict(Xi_val,  verbose=0).ravel()
p_test_cnn = cnn.predict(Xi_test, verbose=0).ravel()
val_scores["CNN"] = evaluate("CNN", y_val, p_val_cnn, store=False, split="Val")
evaluate("CNN", y_test, p_test_cnn)

### 3.4 Classical classifiers on HOG + LBP + GLCM features
To "compare classifiers" we also train three classical models on the combined feature vector, each tuned with **5-fold stratified CV** (`GridSearchCV`, scoring = AUROC) on the training set.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
models = {
    "Logistic Regression": (LogisticRegression(max_iter=3000, class_weight="balanced"),
                            {"C": [0.001, 0.01, 0.1, 1]}),
    "SVM (RBF)":           (SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED),
                            {"C": [1, 10], "gamma": ["scale", 0.0005]}),
    "Random Forest":       (RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=SEED),
                            {"n_estimators": [200, 400], "max_depth": [None, 20], "min_samples_leaf": [1, 3]}),
}
best_models = {}
for name, (est, grid) in models.items():
    t0 = time.time()
    gs = GridSearchCV(est, grid, cv=cv5, scoring="roc_auc", n_jobs=-1).fit(Xf_train, y_train)
    best_models[name] = gs.best_estimator_
    print(f"{name}: best params {gs.best_params_} | CV AUROC {gs.best_score_:.4f} | {time.time()-t0:.0f}s")
    val_scores[name] = evaluate(name, y_val, gs.best_estimator_.predict_proba(Xf_val)[:, 1], store=False, split="Val")
    evaluate(name, y_test, gs.best_estimator_.predict_proba(Xf_test)[:, 1])

### 3.5 Evaluation on the test set – metrics table & ROC curves

In [ ]:
res_df = pd.DataFrame({k: v["metrics"] for k, v in results.items()}).T.round(4)
res_df = res_df.sort_values("AUROC", ascending=False)
display(res_df.style.background_gradient(cmap="YlGn", axis=0).format("{:.4f}"))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
res_df.plot.bar(ax=ax[0], rot=20, ylim=(0.5, 1.0), width=0.8)
ax[0].set_title("Test-set metrics by model"); ax[0].legend(loc="lower right", ncol=2, fontsize=8)
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r["prob"])
    ax[1].plot(fpr, tpr, label=f"{name} (AUC={r['metrics']['AUROC']:.3f})")
ax[1].plot([0, 1], [0, 1], "k--", alpha=.5)
ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate (recall)")
ax[1].set_title("ROC curves (test set)"); ax[1].legend()
plt.tight_layout(); plt.show()

### 3.6 Confusion matrices

In [ ]:
n = len(results); fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4))
for ax, (name, r) in zip(np.atleast_1d(axes), results.items()):
    plot_cm(name, y_test, r["pred"], ax=ax)
plt.tight_layout(); plt.show()

print("Classification report – CNN (test):")
print(classification_report(y_test, results["CNN"]["pred"], target_names=CLASSES, digits=4))

### 3.7 Error analysis – misclassified images

In [ ]:
cnn_pred, cnn_prob = results["CNN"]["pred"], results["CNN"]["prob"]
fp_idx = np.where((cnn_pred == 1) & (y_test == 0))[0]     # Normal predicted as Pneumonia
fn_idx = np.where((cnn_pred == 0) & (y_test == 1))[0]     # Pneumonia missed
print(f"CNN false positives (Normal -> Pneumonia): {len(fp_idx)}")
print(f"CNN false negatives (Pneumonia -> Normal): {len(fn_idx)}")

def show_errors(idxs, title, k=8):
    if len(idxs) == 0: print(f"No {title}"); return
    idxs = idxs[np.argsort(-np.abs(cnn_prob[idxs] - 0.5))][:k]   # most confident mistakes first
    fig, axes = plt.subplots(1, len(idxs), figsize=(2.3 * len(idxs), 3))
    for a, i in zip(np.atleast_1d(axes), idxs):
        a.imshow(X_img[idx_test[i]], cmap="gray"); a.axis("off")
        a.set_title(f"P(pneu)={cnn_prob[i]:.2f}", fontsize=9)
    plt.suptitle(title); plt.tight_layout(); plt.show()

show_errors(fp_idx, "False Positives: truly NORMAL, predicted PNEUMONIA")
show_errors(fn_idx, "False Negatives: truly PNEUMONIA, predicted NORMAL")

# Does the decision threshold trade FN for FP?  (screening usually favours high recall)
rows = []
for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pr = (cnn_prob >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(y_test, pr).ravel()
    rows.append(dict(threshold=thr, FP=fp, FN=fn, Precision=precision_score(y_test, pr), Recall=recall_score(y_test, pr), F1=f1_score(y_test, pr)))
display(pd.DataFrame(rows).round(4))

**Why do misclassifications happen? (typical findings – cross-check with your own confusion matrix & images above)**

* **False positives (Normal → Pneumonia).** Normal X-rays with dense soft-tissue shadows (heart, diaphragm, rib/clavicle overlap), rotated or poorly-inspired films, or exposure/contrast differences look like "opacities". Class weighting deliberately pushes the model toward predicting Pneumonia, which raises FP while lowering FN.
* **False negatives (Pneumonia → Normal).** Early or mild/focal pneumonia has subtle, low-contrast infiltrates; viral pneumonia is more diffuse and less consolidated than bacterial, so it is easier to confuse with a normal lung. Down-sampling to 128×128 can wipe out small findings.
* **Dataset-level causes.** Class imbalance (≈ 1 : 2.9), label noise (labels were derived from reports by several readers), pediatric patients only (varying size/positioning), and scanner/acquisition differences – the model may partly learn artefacts (markers, borders, brightness) rather than pathology.
* **Classical models** (HOG/LBP/GLCM) capture only global texture/edge statistics and cannot model spatial context, so they usually make more errors on subtle cases than the CNN, which learns hierarchical local → global features end-to-end.
* **Mitigation:** lower the decision threshold to raise recall (see the threshold table), use higher resolution, stronger augmentation, transfer learning (e.g. ResNet / DenseNet / EfficientNet), and Grad-CAM to check where the model is looking.

### 3.8 Compare classifiers & identify the best performer

In [ ]:
# Model selection uses the VALIDATION set (test set is only for the final report)
val_df = pd.DataFrame(val_scores).T.round(4).sort_values("AUROC", ascending=False)
print("Validation-set metrics:"); display(val_df)

best_name = val_df.index[0]
print(f"\n>>> Best model selected on validation AUROC: {best_name}")
print("Its TEST performance:")
display(res_df.loc[[best_name]])

summary = pd.concat({"Validation": val_df, "Test": res_df.loc[val_df.index].round(4)}, axis=1)
display(summary)

## Conclusion
* **Data:** 5,863 chest X-rays (Normal vs Pneumonia), imbalanced ≈ 1 : 2.9, pooled and re-split 70/15/15 with stratification (no leakage: scaler fitted on training data only; CV for tuning used training data only).
* **Features:** HOG (shape/edges), LBP (micro-texture) and GLCM (second-order texture) were concatenated into one feature vector per image.
* **Models:** A CNN (tuned by 3-fold CV over learning rate, dropout and width) and three classical classifiers (LR, SVM, RF; tuned by 5-fold CV) were compared on Accuracy, Precision, Recall, F1 and AUROC.
* **Best performer:** printed in section 3.8 (selected on validation AUROC). In typical runs the **CNN** outperforms classical models on AUROC/F1 because it learns task-specific spatial features rather than relying on hand-crafted descriptors; the tables above give the exact numbers of your run.
* **Clinical note:** for screening, prioritise **recall** (few missed pneumonia cases) – tune the threshold accordingly and validate on external data before any real-world use. This model is an educational exercise, not a diagnostic device.

### Possible extensions
Transfer learning (ResNet50/DenseNet121), Grad-CAM explainability, CNN on stacked HOG/LBP maps or CNN-embedding + SVM hybrid, k-fold ensembling, probability calibration, and external validation on another hospital's data.